# 열연 압연 하중 데이터 불러오기

원본: `../main/raw_data/metals.xlsx` (시트 `All Grades Data`)

엑셀이 Strict OOXML 형식이라 `pd.read_excel`이 시트를 읽지 못합니다. XML을 직접 파싱합니다.

In [ ]:
import re
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

NS = "{http://purl.oclc.org/ooxml/spreadsheetml/main}"
XLSX = Path("../main/raw_data/metals.xlsx")


def _col_index(ref):
    n = 0
    for ch in re.match(r"[A-Z]+", ref).group():
        n = n * 26 + ord(ch) - 64
    return n - 1


def load_metals(path=XLSX, sheet="All Grades Data", n_cols=10):
    """Strict OOXML 엑셀의 한 시트를 DataFrame으로 읽는다."""
    with zipfile.ZipFile(path) as z:
        wb = z.read("xl/workbook.xml").decode("utf-8")
        rels = z.read("xl/_rels/workbook.xml.rels").decode("utf-8")
        rid = re.search(r'<sheet name="%s"[^>]*r:id="([^"]+)"' % re.escape(sheet), wb).group(1)
        target = re.search(r'<Relationship [^>]*Id="%s"[^>]*Target="([^"]+)"' % rid, rels)
        target = target.group(1) if target else re.search(r'Target="([^"]+)"[^>]*Id="%s"' % rid, rels).group(1)
        target = target.lstrip("/")
        sheet_path = target if target.startswith("xl/") else "xl/" + target

        shared = [
            "".join(t.text or "" for t in si.iter(NS + "t"))
            for si in ET.fromstring(z.read("xl/sharedStrings.xml")).iter(NS + "si")
        ]

        rows = []
        with z.open(sheet_path) as f:
            for _, el in ET.iterparse(f):
                if el.tag != NS + "row":
                    continue
                row = [None] * n_cols
                for c in el.findall(NS + "c"):
                    i = _col_index(c.get("r"))
                    if i >= n_cols:
                        continue
                    v = c.find(NS + "v")
                    if v is None:
                        continue
                    row[i] = shared[int(v.text)] if c.get("t") == "s" else v.text
                rows.append(row)
                el.clear()

    df = pd.DataFrame(rows[1:], columns=rows[0])
    num_cols = [c for c in df.columns if c not in ("Grade", "Caster ID", "Stand ID")]
    df[num_cols] = df[num_cols].apply(pd.to_numeric)
    df["Coil ID"] = df["Coil ID"].astype(int)
    return df


df = load_metals()
print(df.shape)
df.head()

## 연속성이 깨진 코일 제외

한 코일은 F1~F5가 한 번씩 있어야 하고, 앞 스탠드 출구 두께 = 다음 스탠드 입구 두께, 폭은 코일 안에서 일정해야 합니다.
이를 어기는 코일은 행이 코일 사이에서 밀린 것으로 보고 코일 단위로 통째로 제외합니다.

In [ ]:
STANDS = ["F1", "F2", "F3", "F4", "F5"]
TOL_GAUGE = 0.01  # mm, 앞 스탠드 출구 두께와 다음 스탠드 입구 두께 허용 오차
TOL_WIDTH = 0.01  # mm, 한 코일 안에서 폭 허용 변동


def find_broken_coils(df, tol_gauge=TOL_GAUGE, tol_width=TOL_WIDTH):
    """연속성이 깨진 코일을 사유별로 찾아 {사유: 코일 ID 집합}으로 반환한다."""
    d = df.sort_values(["Coil ID", "Stand ID"])
    g = d.groupby("Coil ID")

    stand_ok = g["Stand ID"].apply(lambda s: list(s) == STANDS)
    incomplete = set(stand_ok.index[~stand_ok])

    gap = (d["Exit gauge (mm)"] - g["Entry gauge (mm)"].shift(-1)).abs()
    gauge_gap = set(d.loc[gap > tol_gauge, "Coil ID"])

    wr = g["Width (mm)"].max() - g["Width (mm)"].min()
    width_var = set(wr.index[wr > tol_width])

    return {"스탠드 구성 불완전": incomplete, "두께 불연속": gauge_gap, "폭 변동": width_var}


reasons = find_broken_coils(df)
for k, v in reasons.items():
    print(f"{k}: {len(v)}개 코일")

broken = set().union(*reasons.values())
df_clean = df[~df["Coil ID"].isin(broken)].reset_index(drop=True)
print(f"\n제외 코일 {len(broken)}개 / 전체 {df['Coil ID'].nunique()}개")
print(f"행: {len(df)} -> {len(df_clean)} ({len(df) - len(df_clean)}행 제외)")
df_clean["Grade"].value_counts().sort_index()

## 데이터 정리 기준 요약

### 1. 원본 데이터 (논문 저자가 이미 정리한 부분)
- 0 이하 값과 결측치 제거 (Grade 2의 코일 2개에서 5행)
- 단위는 전부 SI (하중 N, 길이 mm, 온도 K)

### 2. 우리가 확인만 하고 손대지 않은 것
- 두께 역전(출구 >= 입구): 0건
- 스탠드별 하중 극단값 (중앙값 ± 4.5 x IQR 밖): 0건 -> 제거하지 않음
- 결측치: 0개

### 3. 우리가 제외한 것 (코일 단위로 통째로 제외)
한 코일은 F1~F5가 한 번씩 있어야 하고, 앞 스탠드 출구 두께 = 다음 스탠드 입구 두께, 폭은 코일 안에서 일정해야 한다.
이를 어기는 코일은 행이 코일 사이에서 밀린 것으로 보고 제외한다.

| 기준 | 허용 오차 | 해당 코일 |
|---|---|---|
| 스탠드 구성이 F1~F5 한 번씩이 아님 | - | 8 |
| 앞 스탠드 출구와 다음 스탠드 입구 두께 차이 | 0.01 mm | 249 |
| 코일 내 폭 변동 | 0.01 mm | 261 |
| **합계 (중복 제거)** | | **264 (3.4%)** |

- 결과: 7,797개 -> 7,533개 코일, 38,980행 -> 37,665행 (전부 코일당 5행)
- 행 단위로 고치거나 순서를 복구하지 않음 (틀릴 위험이 있어서)
- 제외 이유: 압하 배분 최적화는 연속성(F1 출구 = F2 입구)을 전제로 하기 때문

### 4. 알려진 영향과 한계
- **Grade05**: 392개 -> 130개 코일 (262개, 67% 제외). 소수 강종이라 성능이 불안정할 수 있음
- Grade02: 1,542개 -> 1,540개. 나머지 강종은 변화 없음
- 제외된 코일은 이상한 코일이라기보다 기록이 밀린 코일이라, 제외로 인한 편향은 Grade05 외에는 작다고 본다
- 속도, 성분, 윤활 정보는 원래 데이터에 없음

### 5. 모델링 때 지킬 것
- 학습/평가는 행이 아니라 **코일 단위로** 분할 (한 코일의 5개 스탠드가 섞이지 않게)
- 강종별 성능을 따로 확인 (Grade05, Grade10 등 소수 강종)
- 최적화는 관측 범위 안으로 제한